##Notebook 8: LLM Orchestration and Report Generation

This notebook builds MeridianIQ's LLM orchestration and report generation layer.

Previous notebooks detected clauses, retrieved evidence, and scored contract risk. This notebook converts those structured outputs into business-readable contract review reports using both deterministic rule-based generation and Gemini-powered LLM generation.

The goal is to ensure that MeridianIQ does not simply produce model outputs, but communicates contract risk clearly, safely, and with supporting evidence.

# Loading Libraries,Risk,Intellignece and Evidence Outputs

In [1]:
import pandas as pd
import numpy as np
import json,joblib
import textwrap
from datetime import datetime
from risk_engine import *

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 300)

In [2]:
risk_scores = pd.read_csv("contract_risk_scores.csv")
risk_drivers = pd.read_csv("risk_driver_table.csv")
contract_intelligence = pd.read_csv("contract_intelligence_table.csv")


detected_clauses = pd.read_csv("sample_detected_clauses.csv")
evidence_chunks = pd.read_csv("sample_evidence_chunks.csv")
predicted_fingerprint = pd.read_csv("sample_predicted_fingerprint.csv")

risk_scores.shape, risk_drivers.shape, contract_intelligence.shape

((510, 5), (472, 8), (11730, 8))

##Recomputing Risk for the Sample Contract

This step applies the risk engine to the predicted fingerprint created from the raw TXT contract.

This ensures that the report is based on the exact contract being reviewed, not only the original CUAD master dataset.

The output includes:

* Risk score
* Risk band
* Triggered risk drivers

In [3]:
risk_config=pd.read_csv("clause_risk_config.csv")
mvp_config = risk_config[risk_config["is_mvp_clause"] == True].copy()

sample_row = predicted_fingerprint.iloc[0]

sample_score, sample_drivers = score_contract(sample_row, mvp_config)
sample_band = assign_risk_band(sample_score)

sample_score, sample_band, len(sample_drivers)

(43, 'Moderate Risk', 2)

##Creating the Sample Risk Driver Table

The risk drivers returned by the risk engine are converted into a DataFrame.

A risk score alone is not enough.
The risk driver table explains which clauses contributed to that score and why they matter.


In [4]:
sample_driver_table = pd.DataFrame(sample_drivers)
sample_driver_table

,clause_name,risk_domain,clause_role,severity,risk_weight,business_description
0,Exclusivity,Competition Risk,Risk if Present,Critical,25,Creates exclusive dealing obligations that may limit business flexibility.
1,Post-Termination Services,Lifecycle Risk,Risk if Present,High,18,Creates obligations that continue after contract termination.


##Adding Business Recommendations

Each triggered risk driver is mapped to a recommended review action.

This turns the risk engine output into something more useful for business users.

Instead of only saying that a risky clause was found, MeridianIQ explains what kind of follow-up action should be considered.

In [5]:
if not sample_driver_table.empty:
    sample_driver_table["recommendation"] = sample_driver_table["clause_name"].map(recommendation_map)

sample_driver_table

,clause_name,risk_domain,clause_role,severity,risk_weight,business_description,recommendation
0,Exclusivity,Competition Risk,Risk if Present,Critical,25,Creates exclusive dealing obligations that may limit business flexibility.,Review exclusivity obligations and assess whether they restrict future business opportunities.
1,Post-Termination Services,Lifecycle Risk,Risk if Present,High,18,Creates obligations that continue after contract termination.,Review post-termination obligations and estimate operational burden.


##Selecting Detected Clauses and Top Evidence

This step filters the detected clause table to keep only clauses predicted as present.

It also selects the top evidence chunks for each clause.

The goal is to keep the report grounded and readable.

In [6]:
detected_present = detected_clauses[detected_clauses["detected"] == True].copy()

top_evidence = (
    evidence_chunks
    .sort_values(["clause_name", "rank"])
    .groupby("clause_name")
    .head(2)
)

##Building the Structured Report Payload

This step creates the central report payload.

This payload is the most important object in the notebook because it separates reasoning from presentation.


In [17]:
report_payload = {
    "report_generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")+ " UTC",
    "filename": predicted_fingerprint.iloc[0]["filename"],
    "risk_score": int(sample_score),
    "risk_band": sample_band,
    "detected_clauses": detected_present["clause_name"].tolist(),
    "risk_drivers": sample_driver_table.to_dict(orient="records"),
    "evidence": top_evidence[
        ["clause_name", "rank", "similarity_score", "evidence_text"]
    ].to_dict(orient="records")
}

report_payload.keys()

dict_keys(['report_generated_at', 'filename', 'risk_score', 'risk_band', 'detected_clauses', 'risk_drivers', 'evidence'])

##Rule-Based Report Generator

Before using Gemini, MeridianIQ creates a deterministic rule-based report.

This is important because the system should still produce a useful output even if an LLM API is unavailable.

The rule-based report converts the structured payload into a readable contract review report

This gives MeridianIQ a reliable fallback reporting layer.

In [18]:
def generate_rule_based_report(payload):
    lines = []

    lines.append("# MeridianIQ Executive Contract Review Report")
    lines.append("")
    lines.append(f"**Generated At:** {payload['report_generated_at']}")
    lines.append(f"**Contract File:** {payload['filename']}")
    lines.append("")
    lines.append("## Overall Risk Assessment")
    lines.append("")
    lines.append(f"**Risk Score:** {payload['risk_score']}/100")
    lines.append(f"**Risk Band:** {payload['risk_band']}")
    lines.append("")

    lines.append("## Detected Clauses")
    lines.append("")
    if payload["detected_clauses"]:
        for clause in payload["detected_clauses"]:
            lines.append(f"- {clause}")
    else:
        lines.append("- No MVP clauses detected.")
    lines.append("")

    lines.append("## Key Risk Drivers")
    lines.append("")
    if payload["risk_drivers"]:
        for driver in payload["risk_drivers"]:
            lines.append(
                f"- **{driver['clause_name']}** "
                f"({driver['severity']}, {driver['risk_domain']}): "
                f"{driver.get('business_description', '')}"
            )
            if driver.get("recommendation"):
                lines.append(f"  - Recommendation: {driver['recommendation']}")
    else:
        lines.append("- No major risk drivers triggered.")
    lines.append("")

    lines.append("## Supporting Evidence")
    lines.append("")
    if payload["evidence"]:
        for item in payload["evidence"]:
            evidence = str(item["evidence_text"])[:700]
            lines.append(f"### {item['clause_name']} — Evidence Rank {item['rank']}")
            lines.append("")
            lines.append(f"> {evidence}...")
            lines.append("")
    else:
        lines.append("- No evidence chunks available.")

    lines.append("")
    lines.append("## Review Note")
    lines.append("")
    lines.append(
        "This report is generated by an AI-assisted contract intelligence system. "
        "It is intended for preliminary review and should not be treated as legal advice."
    )

    return "\n".join(lines)

##Generating the Rule-Based Report

This step runs the rule-based report generator and previews the generated report.

The purpose is to verify that the payload contains enough information to produce a complete business-facing report even without an LLM.

If this report is meaningful, it confirms that MeridianIQ's structured pipeline is strong before Gemini is added on top.

In [19]:
fallback_report = generate_rule_based_report(report_payload)

print(fallback_report[:3000])

# MeridianIQ Executive Contract Review Report

**Generated At:** 2026-06-26 11:58:06 UTC
**Contract File:** ALAMOGORDOFINANCIALCORP_12_16_1999-EX-1-AGENCY AGREEMENT.txt

## Overall Risk Assessment

**Risk Score:** 43/100
**Risk Band:** Moderate Risk

## Detected Clauses

- Non-Compete
- Exclusivity
- Change Of Control
- Anti-Assignment
- Revenue/Profit Sharing
- Minimum Commitment
- Ip Ownership Assignment
- License Grant
- Post-Termination Services
- Audit Rights
- Cap On Liability
- Liquidated Damages
- Insurance

## Key Risk Drivers

- **Exclusivity** (Critical, Competition Risk): Creates exclusive dealing obligations that may limit business flexibility.
  - Recommendation: Review exclusivity obligations and assess whether they restrict future business opportunities.
- **Post-Termination Services** (High, Lifecycle Risk): Creates obligations that continue after contract termination.
  - Recommendation: Review post-termination obligations and estimate operational burden.

## Supporti

##Building the LLM Prompt

This step creates the prompt used for Gemini report generation.

The prompt is carefully constrained with rules.

These guardrails are essential because contract review is a sensitive domain.

The LLM is not treated as the source of truth.
It is used as a communication layer that rewrites structured MeridianIQ outputs into an executive report.

In [10]:
def build_llm_prompt(payload):
    compact_payload = json.dumps(payload, indent=2)[:12000]

    prompt = f"""
You are MeridianIQ, an AI contract review assistant.

Your task is to generate an executive-level contract risk report using ONLY the structured data provided below.

Rules:
- Do not invent clauses, dates, parties, obligations, or risks.
- Use only the detected clauses, risk drivers, recommendations, and evidence provided.
- Do not provide legal advice.
- Write clearly for a business decision-maker.
- If evidence is weak or unavailable, say so.
- Keep the report structured and concise.

Required sections:
1. Executive Summary
2. Overall Risk Assessment
3. Key Risk Drivers
4. Clause Evidence Highlights
5. Recommended Review Actions
6. Disclaimer

Structured Contract Data:
{compact_payload}
"""
    return prompt

##Installing the Gemini SDK

This step installs the Google GenAI SDK required to call Gemini from the notebook.

In [11]:
!pip install -U google-genai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 743.1 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.0/958.0 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.3/252.3 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 26.1 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.4
    Uninstalling pydantic_core-2.41.4:
      Successfully uninstalled pydantic_core-2.41.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.12.3
    Uninstalling pydantic-2.12.3:
      Successfully uninstalled pydantic-2.12.3
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.47.0
    Uninstalling google-auth-2.47.0:
      Successfully uninstalled google-auth-2.47.

##Setting the Gemini API Key

The Gemini API key is entered securely at runtime.

The key is not hardcoded into the notebook, which is important for both security and GitHub readiness.

In [12]:
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Enter Gemini API key: ")

Enter Gemini API key: ··········


##Generating the Gemini Report

This step sends the structured MeridianIQ prompt to Gemini and receives an executive contract review report.

Gemini uses the provided payload, risk drivers, recommendations, and evidence to generate a polished business-facing summary.

This is the LLM orchestration layer of MeridianIQ.

The key point is that Gemini is not independently deciding what is risky.
It is communicating the risk intelligence already produced by the deterministic MeridianIQ pipeline.

In [14]:
from google import genai

client = genai.Client()

llm_prompt = build_llm_prompt(report_payload)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=llm_prompt
)
gemini_report = response.text
print(gemini_report)

**MeridianIQ Contract Risk Report**

**Report Generated:** 2026-06-26 11:43:10
**Contract Name:** ALAMOGORDOFINANCIALCORP_12_16_1999-EX-1-AGENCY AGREEMENT.txt

---

**1. Executive Summary**

This report provides an executive-level risk assessment for the contract "ALAMOGORDOFINANCIALCORP_12_16_1999-EX-1-AGENCY AGREEMENT.txt". The contract has been categorized as **Moderate Risk** with a MeridianIQ risk score of 43. The primary risk drivers identified are clauses related to Exclusivity and Post-Termination Services, which could significantly impact business flexibility and ongoing obligations. A detailed review of these areas is recommended.

---

**2. Overall Risk Assessment**

The contract exhibits a **Moderate Risk** profile with a calculated risk score of 43. This indicates several areas that warrant careful consideration and potential negotiation to mitigate future operational or financial exposure.

---

**3. Key Risk Drivers**

The following clauses represent the most significant

##Exporting Reports and Payloads

In [ ]:
import json

with open("sample_gemini_report.md", "w", encoding="utf-8") as f:
    f.write(gemini_report)

with open("sample_contract_payload.json", "w", encoding="utf-8") as f:
    json.dump(report_payload, f, indent=2)